[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/tutorial-rebuild/tutorial/phase2_public_datasets/02_utilization_lab.ipynb)

> **Run this notebook in Google Colab** — click the badge above. The setup cell auto-clones the repo.


# Phase 2B - Utilization lab

**Phase 2B - The 3-arm utilization lab (independent of retrieval hit).** Independent notebook - runs standalone in Colab or locally (offline, no key).

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "tutorial-rebuild", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

print("SOURCE_DIR =", SOURCE_DIR)

## 📖 Narrative: The 3-arm contrastive reader

This is the **methodological core** of the tutorial. Many benchmarks set
"memory used = evidence retrieved" — making utilization just retrieval in disguise.

Our 3-arm reader avoids this circularity by testing the agent under three conditions:
- **p0** (no context) — baseline: what the agent knows without memory
- **oracle** (gold evidence) — ceiling: the best possible memory
- **provider** (retrieved context) — what the actual system provides

The reader **never sees the gold answer** — it scores choices by token overlap with
the question+context. This is a genuine independent signal.

`provider_rate - p0` is the honest memory gain; `oracle_rate` is the ceiling.

In [ ]:
import pathlib
cands = sorted(pathlib.Path(RESULTS_DIR).glob('run_*_real_utilization_lab.tsv')) \
 + sorted((RESULTS_DIR/'summaries').glob('*utilization_lab.tsv'))
lab = cands[-1] if cands else None
print(lab.name, ':\n'); print(lab.read_text()) if lab else print('Run --mode real first')

## 📖 Narrative: Interpreting the utilization lab

The `net_gain` column shows `provider_rate − p0`. If it's positive, memory **helped**.
If zero, memory was irrelevant. If negative, memory **hurt** (distracting context).

`oracle_rate` is the ceiling — if `provider_rate < oracle_rate`, the system is losing
information in retrieval/storage (not reaching the gold evidence).

### 📝 Quick Quiz
1. **What is the baseline accuracy (p0) across all strategies?** Why is it the same?
2. **Which strategy has the highest `net_gain`?** What does that tell you?
3. **If `provider_rate = oracle_rate`, what does that mean?** Is the system perfect?
4. **The `harm` column shows cases where memory hurt.** How is that possible?

## Probe one failure in the raw trace

In [ ]:
import json
raws = sorted(pathlib.Path(RESULTS_DIR).glob('run_*_locomo_raw.json'))
raw = json.loads(raws[-1].read_text()) if raws else None
if raw:
 rec = next((r for r in raw['records'] if r.get('failure_category')=='retrieval_miss'), raw['records'][0])
 print('Q:', rec['question'][:70]); print('gold:', rec['evidence_ids'][:5]); print('retrieved:', rec['retrieved_memory_ids'][:5])
else: print('(no raw trace - run the diagnose notebook first)')

## 📖 Narrative: Inspecting a real failure

This cell pulls a specific `retrieval_miss` from the raw trace — a question where the
system retrieved the wrong evidence. Compare `gold` (what SHOULD have been retrieved) vs
`retrieved` (what WAS retrieved). The gap is the retrieval failure.

### 📝 Quick Quiz
1. **Look at the gold vs retrieved IDs.** Were any gold IDs in the retrieved set?
2. **Why might the system retrieve the wrong turns?** Think about how TF-IDF matches.
3. **How would a semantic embedding (vs TF-IDF) change this?** Would it help or hurt?